# Generate Weekly Operational Status Feed

## Purpose

This notebook converts the synthetic monthly missed-refund cases into a simulated weekly operational history.

The simulation models:

- monthly case cohorts entering the operational backlog
- variable processing time of up to 10 combined hours per month
- backlog growth and reduction
- case ageing
- dependency cases
- completed cases and final outcomes
- an uneven 80% / 20% completion split between two trained agents

Each output row represents the status of one case on one weekly reporting date.

The combination of `Case ID` and `Reporting Date` uniquely identifies each weekly record.

In [1]:
from pathlib import Path
import random
import sqlite3

import numpy as np
import pandas as pd

## Configuration

The simulation settings are defined in one place so the workflow can be adjusted without changing the main processing logic.

In [2]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Random seed set successfully.")

Random seed set successfully.


In [3]:
CASES_PER_HOUR = 30

MONTHLY_HOUR_OPTIONS = [0, 2, 4, 6, 8, 10]
MONTHLY_HOUR_WEIGHTS = [5, 10, 35, 35, 10, 5]

expected_hours = np.average(
    MONTHLY_HOUR_OPTIONS,
    weights=MONTHLY_HOUR_WEIGHTS,
)

expected_capacity = expected_hours * CASES_PER_HOUR

print(f"Average allocated hours: {expected_hours:.1f}")
print(f"Average monthly capacity: {expected_capacity:.0f} cases")

Average allocated hours: 5.0
Average monthly capacity: 150 cases


## Load Source Case Data

The combined monthly dataset provides the case attributes used to create the weekly operational history.

In [4]:
database_path = Path(
    "../sql/missed_refunds.db"
)

output_path = Path(
    "../data/weekly/weekly_case_status.csv"
)

connection = sqlite3.connect(
    database_path
)

source_query = """
SELECT *
FROM missed_refunds;
"""

source_df = pd.read_sql_query(
    source_query,
    connection,
    parse_dates=["Snapshot Date"],
)

print(f"Source rows: {len(source_df)}")
print(f"Source columns: {len(source_df.columns)}")

source_df.head()

Source rows: 2871
Source columns: 13


,Case ID,Policy Number,Client Number,Snapshot Date,Outstanding Amount,Cancellation Status,Cancelling Department,Cancelling Agent,Agent Working,Refund Type,Root Cause,Customer Contacted,Outcome
0,DH000001,PET000001,CL000029,2025-01-31,3.11,Cancelled,Customer Service,CS018,DH001,Cancellation Before Premium Due,Payment Date Misunderstood,Yes,Actioned
1,DH000002,PET000002,CL000155,2025-01-31,2.39,Cancelled,Customer Service,CS008,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,No Action
2,DH000003,PET000003,CL000179,2025-01-31,21.12,Cancelled,Retentions,RET013,DH001,Death of Pet,Payment Date Misunderstood,Yes,Actioned
3,DH000004,PET000004,CL000032,2025-01-31,175.71,Cancelled,Claims,CLM010,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,Actioned
4,DH000005,PET000005,CL000075,2025-01-31,271.06,Cancelled,Retentions,RET015,DH002,Death of Pet,Agent Forgot,No,Actioned


In [5]:
source_df["Date Added"] = (
    source_df["Snapshot Date"]
    + pd.offsets.MonthBegin(1)
    + pd.Timedelta(days=14)
)

source_df[
    ["Snapshot Date", "Date Added"]
].drop_duplicates().head()

,Snapshot Date,Date Added
0,2025-01-31,2025-02-15
195,2025-02-28,2025-03-15
334,2025-03-31,2025-04-15
467,2025-04-30,2025-05-15
644,2025-05-31,2025-06-15


In [6]:
monthly_arrivals = (
    source_df
    .groupby("Snapshot Date")
    .size()
)

average_monthly_arrivals = monthly_arrivals.mean()

capacity_gap = (
    average_monthly_arrivals
    - expected_capacity
)

print(
    f"Average monthly arrivals: "
    f"{average_monthly_arrivals:.1f} cases"
)

print(
    f"Average capacity gap: "
    f"{capacity_gap:.1f} cases per month"
)

Average monthly arrivals: 159.5 cases
Average capacity gap: 9.5 cases per month


### Capacity Assumption Check

The source data contains an average of **159.5 new cases per month**, while the simulated average processing capacity is **150 cases per month**.

This creates an average shortfall of **9.5 cases per month**. The shortfall, combined with monthly variation in allocated hours, allows realistic backlogs to develop while still permitting occasional periods of backlog reduction.

These values are synthetic operational assumptions and can be adjusted after reviewing the final backlog pattern.

## Create Weekly Reporting Calendar

Weekly reporting dates occur every Friday.

The simulation continues for three months after the final cohort is added so that the remaining backlog can continue to be processed and monitored.

In [7]:
first_date_added = source_df["Date Added"].min()
last_date_added = source_df["Date Added"].max()

simulation_end_date = (
    last_date_added
    + pd.DateOffset(months=3)
)

reporting_dates = pd.date_range(
    start=first_date_added,
    end=simulation_end_date,
    freq="W-FRI",
)

print(f"First reporting date: {reporting_dates.min().date()}")
print(f"Last reporting date: {reporting_dates.max().date()}")
print(f"Number of reporting weeks: {len(reporting_dates)}")

First reporting date: 2025-02-21
Last reporting date: 2026-10-09
Number of reporting weeks: 86


## Generate Monthly Processing Capacity

Each month receives a simulated number of combined hours for the two trained agents.

The allocated hours are converted into the maximum number of cases that can be reviewed during that month.

In [8]:
simulation_months = pd.period_range(
    start=reporting_dates.min().to_period("M"),
    end=reporting_dates.max().to_period("M"),
    freq="M",
)

monthly_capacity_df = pd.DataFrame({
    "Capacity Month": simulation_months,
})

monthly_capacity_df["Allocated Hours"] = np.random.choice(
    MONTHLY_HOUR_OPTIONS,
    size=len(monthly_capacity_df),
    p=np.array(MONTHLY_HOUR_WEIGHTS) / 100,
)

monthly_capacity_df["Monthly Capacity"] = (
    monthly_capacity_df["Allocated Hours"]
    * CASES_PER_HOUR
)

monthly_capacity_df.head()

,Capacity Month,Allocated Hours,Monthly Capacity
0,2025-02,4,120
1,2025-03,10,300
2,2025-04,6,180
3,2025-05,6,180
4,2025-06,4,120


In [9]:
weekly_capacity_df = pd.DataFrame({
    "Reporting Date": reporting_dates,
})

weekly_capacity_df["Capacity Month"] = (
    weekly_capacity_df["Reporting Date"]
    .dt.to_period("M")
)

weekly_capacity_df = weekly_capacity_df.merge(
    monthly_capacity_df,
    on="Capacity Month",
    how="left",
)

weekly_capacity_df.head(10)

,Reporting Date,Capacity Month,Allocated Hours,Monthly Capacity
0,2025-02-21,2025-02,4,120
1,2025-02-28,2025-02,4,120
2,2025-03-07,2025-03,10,300
3,2025-03-14,2025-03,10,300
4,2025-03-21,2025-03,10,300
5,2025-03-28,2025-03,10,300
6,2025-04-04,2025-04,6,180
7,2025-04-11,2025-04,6,180
8,2025-04-18,2025-04,6,180
9,2025-04-25,2025-04,6,180


### Distribute Monthly Hours Across Reporting Weeks

Monthly allocated hours are distributed across Fridays in two-hour blocks.

A reporting week may receive no processing time or multiple two-hour blocks. This preserves the monthly allocation while creating uneven weekly activity.

In [10]:
weekly_capacity_df["Weekly Allocated Hours"] = 0

for _, month_row in monthly_capacity_df.iterrows():

    month_mask = (
        weekly_capacity_df["Capacity Month"]
        == month_row["Capacity Month"]
    )

    available_week_indices = (
        weekly_capacity_df.index[month_mask]
        .to_numpy()
    )

    number_of_blocks = int(
        month_row["Allocated Hours"] / 2
    )

    if number_of_blocks > 0:
        selected_week_indices = np.random.choice(
            available_week_indices,
            size=number_of_blocks,
            replace=True,
        )

        for week_index in selected_week_indices:
            weekly_capacity_df.loc[
                week_index,
                "Weekly Allocated Hours",
            ] += 2

weekly_capacity_df["Weekly Capacity"] = (
    weekly_capacity_df["Weekly Allocated Hours"]
    * CASES_PER_HOUR
)

weekly_capacity_df[
    [
        "Reporting Date",
        "Capacity Month",
        "Weekly Allocated Hours",
        "Weekly Capacity",
    ]
].head(12)

,Reporting Date,Capacity Month,Weekly Allocated Hours,Weekly Capacity
0,2025-02-21,2025-02,0,0
1,2025-02-28,2025-02,4,120
2,2025-03-07,2025-03,0,0
3,2025-03-14,2025-03,2,60
4,2025-03-21,2025-03,2,60
5,2025-03-28,2025-03,6,180
6,2025-04-04,2025-04,0,0
7,2025-04-11,2025-04,4,120
8,2025-04-18,2025-04,2,60
9,2025-04-25,2025-04,0,0


In [11]:
weekly_hour_totals = (
    weekly_capacity_df
    .groupby("Capacity Month")[
        "Weekly Allocated Hours"
    ]
    .sum()
    .reset_index(name="Distributed Hours")
)

capacity_check = monthly_capacity_df[
    ["Capacity Month", "Allocated Hours"]
].merge(
    weekly_hour_totals,
    on="Capacity Month",
    how="left",
)

capacity_check["Hours Match"] = (
    capacity_check["Allocated Hours"]
    == capacity_check["Distributed Hours"]
)

print(
    "All monthly hours preserved:",
    capacity_check["Hours Match"].all(),
)

capacity_check.head()

All monthly hours preserved: True


,Capacity Month,Allocated Hours,Distributed Hours,Hours Match
0,2025-02,4,4,True
1,2025-03,10,10,True
2,2025-04,6,6,True
3,2025-05,6,6,True
4,2025-06,4,4,True


In [12]:
COMPLETING_AGENTS = ["DH001", "DH002"]
AGENT_COMPLETION_WEIGHTS = [80, 20]

DEPENDENCY_STATUSES = [
    "Awaiting Other Department",
    "Awaiting Senior Review",
]

DEPENDENCY_STATUS_WEIGHTS = [85, 15]

OTHER_DEPARTMENT_MIN_DAYS = 1
OTHER_DEPARTMENT_MAX_DAYS = 5
SENIOR_REVIEW_DAYS = 5

## Initialise Case States

A working state table stores the current position of every case during the simulation.

`Not Added` is an internal status used before a case reaches its operational `Date Added`. It will not appear in the final weekly feed.

In [13]:
case_state_df = source_df.copy()

case_state_df["Case Status"] = "Not Added"
case_state_df["Review Date"] = pd.NaT
case_state_df["Completion Date"] = pd.NaT
case_state_df["Completed By"] = pd.NA
case_state_df["Final Outcome"] = pd.NA
case_state_df["Dependency Resolution Date"] = pd.NaT

case_state_df["Case Status"].value_counts()

Case Status
Not Added    2871
Name: count, dtype: int64

In [14]:
def generate_final_outcome(
    source_outcome,
    case_age_days,
):
    """
    Convert the source outcome into a clearer final outcome.
    """

    if source_outcome in ["Actioned", "Raised"]:
        return "Refund Processed"

    if case_age_days <= 14:
        weights = [20, 80]

    elif case_age_days <= 30:
        weights = [35, 65]

    elif case_age_days <= 60:
        weights = [55, 45]

    else:
        weights = [65, 35]

    return random.choices(
        population=[
            "Refund Already Processed",
            "No Refund Due",
        ],
        weights=weights,
        k=1,
    )[0]

In [15]:
def generate_review_date(
    reporting_date,
    date_added,
):
    """
    Select a valid working day after the case
    became available and by the reporting Friday.
    """

    week_start = (
        reporting_date
        - pd.offsets.BDay(4)
    )

    earliest_possible_date = max(
        pd.Timestamp(date_added),
        pd.Timestamp(week_start),
    )

    valid_review_dates = pd.bdate_range(
        start=earliest_possible_date,
        end=reporting_date,
    )

    return random.choice(
        valid_review_dates.tolist()
    )

## Run Weekly Case Simulation

For each Friday reporting date, the simulation will:

1. add newly available cases to the open backlog
2. resolve dependencies whose resolution dates have passed
3. select the oldest open cases up to that week’s capacity
4. complete cases or move raised cases into a dependency status
5. record the status of every active case for that reporting week

In [16]:
def add_new_cases(
    case_states,
    reporting_date,
):
    """
    Move eligible cases from Not Added to Open.
    """

    new_case_mask = (
        (case_states["Case Status"] == "Not Added")
        & (
            case_states["Date Added"]
            <= reporting_date
        )
    )

    case_states.loc[
        new_case_mask,
        "Case Status",
    ] = "Open"

    return int(new_case_mask.sum())

In [17]:
def resolve_dependency_cases(
    case_states,
    reporting_date,
):
    """
    Complete dependency cases that have reached
    their resolution date.
    """

    resolved_mask = (
        case_states["Case Status"].isin(
            DEPENDENCY_STATUSES
        )
        & (
            case_states[
                "Dependency Resolution Date"
            ]
            <= reporting_date
        )
    )

    resolved_indices = case_states.index[
        resolved_mask
    ]

    resolved_count = len(resolved_indices)

    if resolved_count == 0:
        return 0

    case_states.loc[
        resolved_indices,
        "Completion Date",
    ] = case_states.loc[
        resolved_indices,
        "Dependency Resolution Date",
    ]

    case_states.loc[
        resolved_indices,
        "Completed By",
    ] = random.choices(
        population=COMPLETING_AGENTS,
        weights=AGENT_COMPLETION_WEIGHTS,
        k=resolved_count,
    )

    case_states.loc[
        resolved_indices,
        "Final Outcome",
    ] = "Refund Processed"

    case_states.loc[
        resolved_indices,
        "Case Status",
    ] = "Completed"

    return resolved_count

In [18]:
def select_cases_for_review(
    case_states,
    weekly_capacity,
):
    """
    Select the oldest open cases up to the
    available weekly capacity.
    """

    open_cases = (
        case_states[
            case_states["Case Status"] == "Open"
        ]
        .sort_values(
            by=[
                "Date Added",
                "Case ID",
            ]
        )
    )

    selected_cases = open_cases.head(
        int(weekly_capacity)
    )

    return selected_cases.index

In [19]:
def complete_case(
    case_states,
    case_index,
    completion_date,
):
    """
    Record the completion details for one case.
    """

    review_date = case_states.at[
        case_index,
        "Review Date",
    ]

    case_age_days = (
        review_date
        - case_states.at[
            case_index,
            "Date Added",
        ]
    ).days

    source_outcome = case_states.at[
        case_index,
        "Outcome",
    ]

    case_states.at[
        case_index,
        "Completion Date",
    ] = completion_date

    case_states.at[
        case_index,
        "Completed By",
    ] = random.choices(
        population=COMPLETING_AGENTS,
        weights=AGENT_COMPLETION_WEIGHTS,
        k=1,
    )[0]

    case_states.at[
        case_index,
        "Final Outcome",
    ] = generate_final_outcome(
        source_outcome,
        case_age_days,
    )

    case_states.at[
        case_index,
        "Case Status",
    ] = "Completed"

In [20]:
def generate_dependency_details(
    review_date,
):
    """
    Generate a dependency status and resolution date.
    """

    dependency_status = random.choices(
        population=DEPENDENCY_STATUSES,
        weights=DEPENDENCY_STATUS_WEIGHTS,
        k=1,
    )[0]

    if dependency_status == "Awaiting Senior Review":
        resolution_days = SENIOR_REVIEW_DAYS

    else:
        resolution_days = random.randint(
            OTHER_DEPARTMENT_MIN_DAYS,
            OTHER_DEPARTMENT_MAX_DAYS,
        )

    resolution_date = (
        review_date
        + pd.offsets.BDay(resolution_days)
    )

    return dependency_status, resolution_date

In [21]:
def process_selected_cases(
    case_states,
    selected_indices,
    reporting_date,
):
    """
    Review selected cases and either complete them
    or move them into a dependency status.
    """

    completed_count = 0
    dependency_count = 0

    for case_index in selected_indices:

        date_added = case_states.at[
            case_index,
            "Date Added",
        ]

        review_date = generate_review_date(
            reporting_date,
            date_added,
        )

        case_states.at[
            case_index,
            "Review Date",
        ] = review_date

        source_outcome = case_states.at[
            case_index,
            "Outcome",
        ]

        if source_outcome == "Raised":

            (
                dependency_status,
                resolution_date,
            ) = generate_dependency_details(
                review_date
            )

            case_states.at[
                case_index,
                "Dependency Resolution Date",
            ] = resolution_date

            if resolution_date <= reporting_date:
                complete_case(
                    case_states,
                    case_index,
                    resolution_date,
                )

                completed_count += 1

            else:
                case_states.at[
                    case_index,
                    "Case Status",
                ] = dependency_status

                dependency_count += 1

        else:
            complete_case(
                case_states,
                case_index,
                review_date,
            )

            completed_count += 1

    return completed_count, dependency_count

In [22]:
def create_weekly_snapshot(
    case_states,
    reporting_date,
    previous_reporting_date,
):
    """
    Create the weekly output rows for active cases
    and cases completed during the reporting week.
    """

    unresolved_mask = case_states[
        "Case Status"
    ].isin(
        [
            "Open",
            *DEPENDENCY_STATUSES,
        ]
    )

    completed_this_week_mask = (
        (case_states["Case Status"] == "Completed")
        & (
            case_states["Completion Date"]
            > previous_reporting_date
        )
        & (
            case_states["Completion Date"]
            <= reporting_date
        )
    )

    snapshot_mask = (
        unresolved_mask
        | completed_this_week_mask
    )

    weekly_snapshot = case_states.loc[
        snapshot_mask,
        [
            "Case ID",
            "Policy Number",
            "Client Number",
            "Outstanding Amount",
            "Cancellation Status",
            "Date Added",
            "Case Status",
            "Completion Date",
            "Completed By",
            "Final Outcome",
        ],
    ].copy()

    weekly_snapshot["Reporting Date"] = (
        reporting_date
    )

    age_end_date = (
        weekly_snapshot["Completion Date"]
        .fillna(reporting_date)
    )

    weekly_snapshot["Case Age Days"] = (
        age_end_date
        - weekly_snapshot["Date Added"]
    ).dt.days

    return weekly_snapshot[
        [
            "Case ID",
            "Policy Number",
            "Client Number",
            "Outstanding Amount",
            "Cancellation Status",
            "Date Added",
            "Reporting Date",
            "Case Status",
            "Completion Date",
            "Completed By",
            "Final Outcome",
            "Case Age Days",
        ]
    ]

In [23]:
weekly_history = []
weekly_summary = []

previous_reporting_date = (
    reporting_dates.min()
    - pd.Timedelta(days=7)
)

for _, week_row in weekly_capacity_df.iterrows():

    reporting_date = week_row[
        "Reporting Date"
    ]

    weekly_capacity = int(
        week_row["Weekly Capacity"]
    )

    new_cases = add_new_cases(
        case_state_df,
        reporting_date,
    )

    resolved_dependencies = (
        resolve_dependency_cases(
            case_state_df,
            reporting_date,
        )
    )

    selected_indices = (
        select_cases_for_review(
            case_state_df,
            weekly_capacity,
        )
    )

    (
        completed_during_review,
        new_dependencies,
    ) = process_selected_cases(
        case_state_df,
        selected_indices,
        reporting_date,
    )

    weekly_snapshot = create_weekly_snapshot(
        case_state_df,
        reporting_date,
        previous_reporting_date,
    )

    weekly_history.append(
        weekly_snapshot
    )

    open_backlog = (
        case_state_df["Case Status"]
        == "Open"
    ).sum()

    dependency_backlog = (
        case_state_df["Case Status"]
        .isin(DEPENDENCY_STATUSES)
    ).sum()

    weekly_summary.append({
        "Reporting Date": reporting_date,
        "New Cases": new_cases,
        "Weekly Capacity": weekly_capacity,
        "Cases Reviewed": len(selected_indices),
        "Cases Completed": (
            completed_during_review
            + resolved_dependencies
        ),
        "New Dependencies": new_dependencies,
        "Open Backlog": open_backlog,
        "Dependency Backlog": dependency_backlog,
        "Total Backlog": (
            open_backlog
            + dependency_backlog
        ),
    })

    previous_reporting_date = reporting_date

In [24]:
weekly_history_df = pd.concat(
    weekly_history,
    ignore_index=True,
)

weekly_summary_df = pd.DataFrame(
    weekly_summary
)

print(
    f"Weekly history rows: "
    f"{len(weekly_history_df)}"
)

print(
    f"Reporting weeks: "
    f"{len(weekly_summary_df)}"
)

weekly_summary_df.tail(10)

Weekly history rows: 15967
Reporting weeks: 86


,Reporting Date,New Cases,Weekly Capacity,Cases Reviewed,Cases Completed,New Dependencies,Open Backlog,Dependency Backlog,Total Backlog
76,2026-08-07,0,0,0,0,0,257,0,257
77,2026-08-14,0,60,60,60,0,197,0,197
78,2026-08-21,0,0,0,0,0,197,0,197
79,2026-08-28,0,60,60,59,1,137,1,138
80,2026-09-04,0,60,60,60,1,77,1,78
81,2026-09-11,0,0,0,1,0,77,0,77
82,2026-09-18,0,60,60,60,0,17,0,17
83,2026-09-25,0,0,0,0,0,17,0,17
84,2026-10-02,0,0,0,0,0,17,0,17
85,2026-10-09,0,180,17,17,0,0,0,0


## Validate Weekly Simulation

The completed weekly history is checked for unique records, complete case coverage, valid ages and final backlog clearance.

In [25]:
peak_backlog_row = weekly_summary_df.loc[
    weekly_summary_df["Total Backlog"].idxmax()
]

print(
    "Unique cases in history:",
    weekly_history_df["Case ID"].nunique(),
)

print(
    "Duplicate weekly records:",
    weekly_history_df.duplicated(
        subset=[
            "Case ID",
            "Reporting Date",
        ]
    ).sum(),
)

print(
    "Cases completed:",
    (
        case_state_df["Case Status"]
        == "Completed"
    ).sum(),
)

print(
    "Final backlog:",
    weekly_summary_df[
        "Total Backlog"
    ].iloc[-1],
)

print(
    "Peak backlog:",
    peak_backlog_row["Total Backlog"],
    "on",
    peak_backlog_row[
        "Reporting Date"
    ].date(),
)

print(
    "Case age range:",
    weekly_history_df[
        "Case Age Days"
    ].min(),
    "to",
    weekly_history_df[
        "Case Age Days"
    ].max(),
    "days",
)

Unique cases in history: 2871
Duplicate weekly records: 0
Cases completed: 2871
Final backlog: 0
Peak backlog: 354 on 2026-06-19
Case age range: 0 to 86 days


In [26]:
agent_completion_summary = (
    case_state_df["Completed By"]
    .value_counts()
    .rename_axis("Completed By")
    .reset_index(name="Cases")
)

agent_completion_summary["Percentage"] = (
    agent_completion_summary["Cases"]
    / agent_completion_summary["Cases"].sum()
    * 100
).round(1)

final_outcome_summary = (
    case_state_df
    .groupby(
        [
            "Outcome",
            "Final Outcome",
        ]
    )
    .size()
    .reset_index(name="Cases")
)

print("Completing agent distribution:")
display(agent_completion_summary)

print("Source-to-final outcome mapping:")
display(final_outcome_summary)

Completing agent distribution:


,Completed By,Cases,Percentage
0,DH001,2332,81.2
1,DH002,539,18.8


Source-to-final outcome mapping:


,Outcome,Final Outcome,Cases
0,Actioned,Refund Processed,2359
1,No Action,No Refund Due,258
2,No Action,Refund Already Processed,205
3,Raised,Refund Processed,49


## Export Weekly Operational Data

The detailed weekly case history and weekly summary are exported as CSV files for SQL and Power BI reporting.

In [27]:
summary_output_path = Path(
    "../data/weekly/weekly_operational_summary.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

weekly_history_df.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

weekly_summary_df.to_csv(
    summary_output_path,
    index=False,
    date_format="%Y-%m-%d",
)

print(f"Saved: {output_path}")
print(f"Saved: {summary_output_path}")

Saved: ..\data\weekly\weekly_case_status.csv
Saved: ..\data\weekly\weekly_operational_summary.csv


## Load Weekly Data into SQLite

The detailed weekly history and weekly summary are written to SQLite so they can be analysed using SQL and connected to Power BI.

In [28]:
weekly_history_df.to_sql(
    name="weekly_case_status",
    con=connection,
    if_exists="replace",
    index=False,
)

weekly_summary_df.to_sql(
    name="weekly_operational_summary",
    con=connection,
    if_exists="replace",
    index=False,
)

print("Created table: weekly_case_status")
print("Created table: weekly_operational_summary")

Created table: weekly_case_status
Created table: weekly_operational_summary


In [29]:
verification_query = """
SELECT
    'weekly_case_status' AS table_name,
    COUNT(*) AS total_rows
FROM weekly_case_status

UNION ALL

SELECT
    'weekly_operational_summary' AS table_name,
    COUNT(*) AS total_rows
FROM weekly_operational_summary;
"""

pd.read_sql_query(
    verification_query,
    connection,
)

,table_name,total_rows
0,weekly_case_status,15967
1,weekly_operational_summary,86


In [30]:
print(
    "DataFrame rows:",
    len(weekly_history_df),
)

print(
    "SQLite rows:",
    pd.read_sql_query(
        "SELECT COUNT(*) AS rows FROM weekly_case_status;",
        connection,
    )["rows"].iloc[0],
)

DataFrame rows: 15967
SQLite rows: 15967


In [31]:
connection.close()

print("Database connection closed.")

Database connection closed.


## Simulation Summary

The weekly operational simulation produced:

- **15,967 weekly case-status records**
- **86 Friday reporting periods**
- **2,871 unique cases**
- no duplicate `Case ID` and `Reporting Date` combinations
- a peak backlog of **354 cases on 19 June 2026**
- a maximum case age of **86 days**
- a final backlog of zero on **9 October 2026**
- an **81.2% / 18.8%** completion split between the two trained agents

The source data ends with the June 2026 monthly snapshot, which enters the operational workflow on 15 July 2026. Reporting continues for a three-month run-off period so that unresolved cases can be monitored until the backlog clears. No new cases are introduced during this run-off period.

The simulation demonstrates how inconsistent processing time can create and reduce operational backlogs even when the underlying monthly case volumes remain relatively stable.

All workload, capacity and completion patterns are synthetic assumptions created for portfolio demonstration.